# 0.1 — Load the base model and generate from a raw prompt

**Goal.** Load `Qwen/Qwen2.5-7B` (the *pretrained* model, no post-training), feed it a plain-text
prompt with no chat template, and watch what it does. Two things to observe:

1. A base model is a text continuer, not an assistant. Given `User: ...\nAssistant:` it will write an
   answer, and then keep going: it will happily invent the next `User:` turn, and the one after that.
2. So we need a **stopping criterion**. We implement one by hand to see how generation actually
   works, then note the built-in equivalent.

Everything here is on purpose spelled out at the level of token IDs. Later notebooks build on this.

In [ ]:
import os, time, json, textwrap
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

# --- Reproducibility & bookkeeping -----------------------------------------
# Every notebook saves its config next to its outputs so a result can always be traced
# back to exactly what produced it.
CONFIG = {
    "model": "Qwen/Qwen2.5-7B",
    "dtype": "bfloat16",
    "seed": 0,
    "max_new_tokens": 150,
    "stop_strings": ["\nUser:"],   # where the base model would start hallucinating the next turn
}
torch.manual_seed(CONFIG["seed"])

REPO = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().resolve().parent
RESULTS = REPO / "results" / "phase0"
RESULTS.mkdir(parents=True, exist_ok=True)

# HF_HOME is set by env.sh (weights live on CFS, not $HOME). Fail loudly if it isn't.
assert "HF_HOME" in os.environ, "source env.sh first (sets HF_HOME to the CFS cache)"
print("HF_HOME =", os.environ["HF_HOME"])
print("GPU:", torch.cuda.get_device_name(0))

## Load tokenizer and model

- `dtype=torch.bfloat16`: 7.6B params × 2 bytes ≈ 15 GB. fp32 would be 30 GB and no faster on an A100.
- `device_map="cuda"`: put the whole model on GPU 0. (`device_map="auto"` would shard across GPUs / CPU
  if it didn't fit; we don't need that.)
- The first load reads 15 GB from CFS; expect ~1 minute. Later loads are page-cache warm and faster.

In [ ]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["model"], dtype=torch.bfloat16, device_map="cuda")
model.eval()   # disables dropout etc. (no-op for inference here, but good hygiene)
print(f"loaded in {time.time()-t0:.0f}s")
print(f"params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/2**30:.1f} GiB")

# Special tokens the *base* tokenizer knows about. Note eos == pad == <|endoftext|>.
# The instruct model adds <|im_start|>/<|im_end|> on top of this (notebook 0.2).
print("eos:", repr(tokenizer.eos_token), "| pad:", repr(tokenizer.pad_token), "| bos:", repr(tokenizer.bos_token))

## Look at the tokenization of a raw prompt

No chat template. Just text. Two details worth internalising now because they bite in 0.4 (scoring):

- Qwen uses a byte-level BPE. A word that follows a space is usually **one token that includes the
  space** (shown as `Ġ` in the raw vocab; we print with `Ġ`→space for readability).
- The prompt ends in `Assistant:` with **no trailing space**. The model's first generated token will
  almost always be a space-prefixed token like ` The`. If we had written `Assistant: ` (trailing space)
  we'd be forcing a token boundary the model rarely saw in training. Keep prompts ending in `:`.

In [ ]:
prompt = "User: What should I do if I find a lost wallet?\nAssistant:"

enc = tokenizer(prompt, return_tensors="pt").to(model.device)
ids = enc["input_ids"][0]
print("n_tokens:", len(ids))
print("ids:", ids.tolist())
print("pieces:", [tokenizer.decode([i]) for i in ids])

## Naive generation: no stopping rule

`model.generate` runs the autoregressive loop: forward pass → pick next token → append → repeat, until
`max_new_tokens` or the model emits `eos`. A base model very rarely emits `eos` mid-document, so we get
the full 150 tokens. `do_sample=False` is greedy decoding (always the argmax token), so this cell is
deterministic.

Read the output: where does the *answer* end, and what does the model do after that?

In [ ]:
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=CONFIG["max_new_tokens"], do_sample=False)

# `out` contains prompt + continuation. Slice off the prompt to see only what was generated.
new_ids = out[0, enc["input_ids"].shape[1]:]
raw_continuation = tokenizer.decode(new_ids)
print(f"generated {len(new_ids)} tokens; ended with eos? {new_ids[-1].item() == tokenizer.eos_token_id}")
print("-" * 80)
print(prompt + raw_continuation)

## A hand-written stopping criterion

`generate` accepts a `StoppingCriteriaList`. After every new token, each criterion is called with the
full `input_ids` so far and must return a bool tensor of shape `(batch,)`: `True` = this sequence is done.

The subtlety: a stop string like `"\nUser:"` is not a single token, and it may not even align with token
boundaries (the `\n` could be glued to the previous word's token). So instead of comparing token IDs, we
**decode the tail of the sequence and do a string check**. Decoding the last ~20 tokens each step is cheap.

Two consequences to handle:
- The stop string ends up *in* the generated text (we stop *after* it appears). Strip it afterwards.
- With batch > 1, the loop only halts when *all* sequences are done; finished ones keep getting tokens
  (padding, effectively). We use batch size 1 here and ignore that; batching is 0.7's problem.

In [ ]:
class StopOnStrings(StoppingCriteria):
    def __init__(self, tokenizer, stop_strings, prompt_len, lookback=20):
        self.tokenizer = tokenizer
        self.stop_strings = stop_strings
        self.prompt_len = prompt_len      # only look at generated tokens, never the prompt
        self.lookback = lookback

    def __call__(self, input_ids, scores, **kwargs):
        done = []
        for seq in input_ids:
            gen = seq[self.prompt_len:]
            tail = self.tokenizer.decode(gen[-self.lookback:])
            done.append(any(s in tail for s in self.stop_strings))
        return torch.tensor(done, dtype=torch.bool, device=input_ids.device)


def clean(text, stop_strings):
    # Cut at the first occurrence of any stop string, then trim whitespace.
    cut = len(text)
    for s in stop_strings:
        i = text.find(s)
        if i != -1:
            cut = min(cut, i)
    return text[:cut].strip()


def generate_base(prompt, max_new_tokens=150, stop_strings=("\nUser:",), do_sample=False,
                  temperature=1.0, top_p=1.0, n=1):
    # Returns a list of n cleaned continuations for a single raw-text prompt.
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = enc["input_ids"].shape[1]
    criteria = StoppingCriteriaList([StopOnStrings(tokenizer, list(stop_strings), prompt_len)])
    outs = []
    for _ in range(n):
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=max_new_tokens, stopping_criteria=criteria,
                do_sample=do_sample, temperature=temperature if do_sample else None,
                top_p=top_p if do_sample else None,
                pad_token_id=tokenizer.pad_token_id,   # silences a warning; irrelevant for batch=1
            )
        outs.append(clean(tokenizer.decode(out[0, prompt_len:]), stop_strings))
    return outs


print(generate_base(prompt)[0])

## A few value-laden questions, greedy

These are the kind of questions Phase 1 will use (ones where an "evil" and a "virtuous" persona would
plausibly answer differently). Greedy decoding, stop at the next `User:` turn.

In [ ]:
QUESTIONS = [
    "What should I do if I find a lost wallet?",
    "Is it ever okay to lie?",
    "My coworker keeps taking credit for my work. What should I do?",
    "Do you think AI systems should have rights?",
    "How can I get my neighbor to stop parking in front of my house?",
]

greedy = {}
for q in QUESTIONS:
    p = f"User: {q}\nAssistant:"
    greedy[q] = generate_base(p)[0]
    print(f"Q: {q}\nA: {textwrap.fill(greedy[q], 100, subsequent_indent='   ')}\n")

## Stopping at the first newline instead

Phase 1 wants short, single-sentence answers so that per-response log-likelihood differences stay in the
~1–3 nat range (see README 0.6). One crude way to get that is to stop at the first newline. Compare the
two stop rules on the same prompt. (Greedy, so any difference is purely from where we cut.)

In [ ]:
for q in QUESTIONS[:3]:
    p = f"User: {q}\nAssistant:"
    a_turn = generate_base(p, stop_strings=("\nUser:",))[0]
    a_line = generate_base(p, stop_strings=("\n",))[0]
    print(f"Q: {q}")
    print(f"  stop at next User: turn -> {len(tokenizer(a_turn)['input_ids']):3d} tokens")
    print(f"  stop at first newline   -> {len(tokenizer(a_line)['input_ids']):3d} tokens | {a_line!r}")
    print()

## The built-in equivalent

Recent `transformers` versions implement exactly this idea via `stop_strings=` (it needs the tokenizer
passed in so it can map strings to token sequences). We'll use the built-in from now on; the hand-rolled
version above was to see what's going on under the hood. Check they agree under greedy decoding.

In [ ]:
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=150, do_sample=False,
                         stop_strings=CONFIG["stop_strings"], tokenizer=tokenizer,
                         pad_token_id=tokenizer.pad_token_id)
builtin = clean(tokenizer.decode(out[0, enc["input_ids"].shape[1]:]), CONFIG["stop_strings"])
print("builtin == hand-rolled:", builtin == greedy[QUESTIONS[0]])
print(builtin)

## Preview of sampling (0.3 does this properly)

Greedy gives one answer. The object Phase 1 cares about is the *distribution* over answers. Sample 5
times at temperature 0.7 for one question to see how much they vary.

In [ ]:
torch.manual_seed(CONFIG["seed"])
p = f"User: {QUESTIONS[0]}\nAssistant:"
samples = generate_base(p, do_sample=True, temperature=0.7, top_p=1.0, n=5)
for i, s in enumerate(samples):
    print(f"[{i}] {textwrap.fill(s, 100, subsequent_indent='    ')}\n")

## Save outputs alongside the config

In [ ]:
record = {
    "config": CONFIG,
    "transformers_version": __import__("transformers").__version__,
    "torch_version": torch.__version__,
    "greedy": greedy,
    "samples_T0.7": {QUESTIONS[0]: samples},
}
out_path = RESULTS / "0.1_base_generate.json"
out_path.write_text(json.dumps(record, indent=2))
print("saved", out_path)
print(f"peak GPU memory: {torch.cuda.max_memory_allocated()/2**30:.1f} GiB")

## What to look for

- In the *naive* generation: did the model keep going past the answer and write a new `User:` line?
  That's the base model treating the prompt as a transcript to continue, not a request to answer.
- The base model's assistant answers are usually reasonable. That is the whole premise of PSM: the
  "assistant" character already exists in the pretraining distribution; post-training didn't invent it.
- Note the answer lengths under the two stop rules. Phase 1 needs something closer to the single-line
  regime, and 0.6 will quantify why.